# Data, state and time alignment

Validate OHLC, construct time bars and persistent IMOM/IMACD states, and align state availability with execution time.

**Part 1/4 of a shared-state notebook series.** This part uses variables/functions from earlier parts. Opening another notebook normally creates a new kernel. Use `python scripts/run_series.py all_for_one --export outputs/all_for_one_combined.ipynb` from the repository root, then open the exported notebook for interactive execution. Exporting does not execute the research or restore missing data.

**Status:** curated research source; original saved outputs preserved. Original notes about results are historical observations, not revalidated conclusions. Read `docs/DATA_AND_RUNNING.md` before execution.

Source: `ALL_FOR_ONE.ipynb`, original cells 0–49 (zero-based).


In [ ]:
from pathlib import Path
import os
import sys

# Locate the repository from its root or any notebook subdirectory.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "tradinguni_paths.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the TradingUni repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from tradinguni_paths import data_path, output_path
os.chdir(REPO_ROOT)


In [1]:
import polars as pl

aggbars2020 = pl.read_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/aggBars/aggBars-1m-ETH-2020.parquet")))
aggbars2021 = pl.read_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/aggBars/aggBars-1m-ETH-2021.parquet")))
aggbars2022 = pl.read_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/aggBars/aggBars-1m-ETH-2022.parquet")))
aggbars2023 = pl.read_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/aggBars/aggBars-1m-ETH-2023.parquet")))
aggbars2024 = pl.read_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/aggBars/aggBars-1m-ETH-2024.parquet")))
aggbars2025 = pl.read_parquet(str(data_path("0.Data -> Facts/Binance Vision Data/aggBars/aggBars-1m-ETH-2025.parquet")))

aggbars = pl.concat([aggbars2020, aggbars2021, aggbars2022, aggbars2023, aggbars2024, aggbars2025])

In [2]:
df = aggbars[['t',	'open',	'high',	'low',	'close']].to_pandas()

In [3]:
df

,t,open,high,low,close
0,2020-01-01 00:00:00,129.16,129.19,128.96,129.02
1,2020-01-01 00:01:00,128.99,129.11,128.99,129.02
2,2020-01-01 00:02:00,129.00,129.05,128.94,128.98
3,2020-01-01 00:03:00,128.95,129.02,128.94,128.98
4,2020-01-01 00:04:00,128.97,128.97,128.80,128.88
...,...,...,...,...,...
3156475,2025-12-31 23:55:00,2971.14,2971.82,2971.13,2971.82
3156476,2025-12-31 23:56:00,2971.82,2972.83,2971.81,2972.82
3156477,2025-12-31 23:57:00,2972.83,2972.83,2971.64,2971.64
3156478,2025-12-31 23:58:00,2971.64,2971.65,2971.64,2971.65


In [4]:
import warnings
# Suppress all warnings globally
warnings.filterwarnings('ignore')
warnings.warn("This will not be displayed", category=UserWarning)
print("Script continues without showing warnings.")

Script continues without showing warnings.


# Data Validation & TF sperate

In [5]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd


REQUIRED_PRICE_COLUMNS = ["t", "open", "high", "low", "close"]


@dataclass(frozen=True)
class PriceValidationReport:
    n_rows_input: int
    n_rows_output: int

    start_time: pd.Timestamp
    end_time: pd.Timestamp

    duplicate_rows_removed: int
    invalid_ohlc_rows_removed: int

    expected_minute_slots: int
    observed_minute_slots: int
    missing_minute_slots: int

    missing_ratio: float
    is_monotonic: bool

In [6]:
def prepare_1m_price_data(
    df: pd.DataFrame,
    *,
    remove_invalid_ohlc: bool = True,
) -> tuple[pd.DataFrame, PriceValidationReport]:
    """
    เตรียมข้อมูล OHLC 1 นาทีให้เป็น canonical dataset

    กฎ:
    - t เป็น UTC
    - เรียงตามเวลา
    - timestamp ไม่ซ้ำ
    - ราคาต้องมากกว่า 0
    - high ต้องไม่น้อยกว่า open/close/low
    - low ต้องไม่มากกว่า open/close/high
    - ไม่ reindex ช่องเวลาที่หาย
    """

    missing_columns = [
        col for col in REQUIRED_PRICE_COLUMNS
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {missing_columns}"
        )

    n_rows_input = len(df)

    d = df[REQUIRED_PRICE_COLUMNS].copy()

    # รองรับทั้ง datetime เดิม, millisecond integer และ string
    if pd.api.types.is_numeric_dtype(d["t"]):
        d["t"] = pd.to_datetime(
            d["t"],
            unit="ms",
            utc=True,
            errors="coerce",
        )
    else:
        d["t"] = pd.to_datetime(
            d["t"],
            utc=True,
            errors="coerce",
        )

    for col in ["open", "high", "low", "close"]:
        d[col] = pd.to_numeric(d[col], errors="coerce")

    # ลบแถวที่ timestamp หรือราคาไม่สมบูรณ์
    d = d.dropna(
        subset=["t", "open", "high", "low", "close"]
    )

    # เรียงก่อนลบ duplicate เพื่อให้ keep="last" มีความหมายชัดเจน
    d = d.sort_values("t", kind="stable")

    duplicate_mask = d["t"].duplicated(keep="last")
    duplicate_rows_removed = int(duplicate_mask.sum())

    d = d.loc[~duplicate_mask].copy()

    positive_price = (
        (d["open"] > 0)
        & (d["high"] > 0)
        & (d["low"] > 0)
        & (d["close"] > 0)
    )

    valid_high = d["high"] >= d[
        ["open", "low", "close"]
    ].max(axis=1)

    valid_low = d["low"] <= d[
        ["open", "high", "close"]
    ].min(axis=1)

    valid_ohlc = positive_price & valid_high & valid_low
    invalid_ohlc_rows_removed = int((~valid_ohlc).sum())

    if invalid_ohlc_rows_removed > 0:
        if remove_invalid_ohlc:
            d = d.loc[valid_ohlc].copy()
        else:
            invalid_rows = d.loc[
                ~valid_ohlc,
                REQUIRED_PRICE_COLUMNS,
            ].head(10)

            raise ValueError(
                "Invalid OHLC rows found.\n"
                f"{invalid_rows}"
            )

    d = (
        d.sort_values("t", kind="stable")
        .reset_index(drop=True)
    )

    if len(d) == 0:
        raise ValueError("No valid price rows remain.")

    start_time = d["t"].iloc[0]
    end_time = d["t"].iloc[-1]

    expected_minute_slots = int(
        (end_time - start_time) / pd.Timedelta(minutes=1)
    ) + 1

    observed_minute_slots = len(d)

    missing_minute_slots = max(
        expected_minute_slots - observed_minute_slots,
        0,
    )

    missing_ratio = (
        missing_minute_slots / expected_minute_slots
        if expected_minute_slots > 0
        else np.nan
    )

    is_monotonic = bool(d["t"].is_monotonic_increasing)

    report = PriceValidationReport(
        n_rows_input=n_rows_input,
        n_rows_output=len(d),
        start_time=start_time,
        end_time=end_time,
        duplicate_rows_removed=duplicate_rows_removed,
        invalid_ohlc_rows_removed=invalid_ohlc_rows_removed,
        expected_minute_slots=expected_minute_slots,
        observed_minute_slots=observed_minute_slots,
        missing_minute_slots=missing_minute_slots,
        missing_ratio=missing_ratio,
        is_monotonic=is_monotonic,
    )

    return d, report

In [7]:
@dataclass(frozen=True)
class TimeBarConfig:
    timeframe: str

    # fixed UTC grid
    origin: str = "epoch"
    offset: str = "0min"

    keep_incomplete_bars: bool = True

In [8]:
def timeframe_to_expected_minutes(timeframe: str) -> int:
    delta = pd.Timedelta(timeframe)
    one_minute = pd.Timedelta(minutes=1)

    ratio = delta / one_minute

    if not float(ratio).is_integer():
        raise ValueError(
            f"Timeframe must be a whole number of minutes: {timeframe}"
        )

    expected_minutes = int(ratio)

    if expected_minutes <= 0:
        raise ValueError(
            f"Invalid timeframe: {timeframe}"
        )

    return expected_minutes

In [9]:
def build_time_bars(
    price_1m: pd.DataFrame,
    cfg: TimeBarConfig,
) -> pd.DataFrame:
    """
    สร้าง OHLC bars จาก canonical 1-minute data

    Semantics:
    - closed='left'
    - label='right'
    - timestamp คือเวลาปิดแท่ง
    - available_time เท่ากับ bar_end
    """

    required = {"t", "open", "high", "low", "close"}

    missing = required.difference(price_1m.columns)
    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    if not price_1m["t"].is_monotonic_increasing:
        raise ValueError(
            "price_1m must be sorted by t."
        )

    if price_1m["t"].duplicated().any():
        raise ValueError(
            "price_1m contains duplicated timestamps."
        )

    expected_minutes = timeframe_to_expected_minutes(
        cfg.timeframe
    )

    x = (
        price_1m[
            ["t", "open", "high", "low", "close"]
        ]
        .set_index("t")
    )

    resampler = x.resample(
        cfg.timeframe,
        origin=cfg.origin,
        offset=cfg.offset,
        closed="left",
        label="right",
    )

    bars = resampler.agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        close=("close", "last"),
        minute_count=("close", "count"),
    )

    # ลบช่วงที่ไม่มีข้อมูลเลย
    bars = bars.loc[bars["minute_count"] > 0].copy()

    bars.index.name = "bar_end"

    bars["bar_start"] = (
        bars.index - pd.Timedelta(cfg.timeframe)
    )

    bars["available_time"] = bars.index

    bars["expected_minute_count"] = expected_minutes

    bars["coverage_ratio"] = (
        bars["minute_count"] / expected_minutes
    )

    bars["is_complete"] = (
        bars["minute_count"] == expected_minutes
    )

    bars["timeframe"] = cfg.timeframe

    # เรียง column ให้อ่านง่าย
    bars = bars[
        [
            "bar_start",
            "open",
            "high",
            "low",
            "close",
            "minute_count",
            "expected_minute_count",
            "coverage_ratio",
            "is_complete",
            "available_time",
            "timeframe",
        ]
    ]

    if not cfg.keep_incomplete_bars:
        bars = bars.loc[bars["is_complete"]].copy()

    return bars

In [10]:
REGIME_TIMEFRAMES = [
    "15min",
    "1h",
    "4h",
    "6h",
    "16h",
    "24h",
]


def build_bar_library(
    price_1m: pd.DataFrame,
    timeframes: Iterable[str],
    *,
    keep_incomplete_bars: bool = True,
) -> dict[str, pd.DataFrame]:

    result: dict[str, pd.DataFrame] = {}

    for timeframe in timeframes:
        cfg = TimeBarConfig(
            timeframe=timeframe,
            keep_incomplete_bars=keep_incomplete_bars,
        )

        result[timeframe] = build_time_bars(
            price_1m,
            cfg,
        )

    return result

In [11]:
# for timeframe, bars in bar_library.items():
#     print("=" * 80)
#     print("Timeframe:", timeframe)
#     print("Shape:", bars.shape)
#     print("Complete ratio:", bars["is_complete"].mean())
#     print("Mean coverage:", bars["coverage_ratio"].mean())
#     print("Start:", bars.index.min())
#     print("End:", bars.index.max())

#     display(bars.head(3))
#     display(bars.tail(3))

In [12]:
def validate_time_bars(
    bars: pd.DataFrame,
    timeframe: str,
) -> None:

    if not bars.index.is_monotonic_increasing:
        raise AssertionError(
            f"{timeframe}: bar index is not monotonic."
        )

    if bars.index.duplicated().any():
        raise AssertionError(
            f"{timeframe}: duplicated bar_end."
        )

    expected_duration = pd.Timedelta(timeframe)

    actual_duration = (
        bars.index.to_series()
        - bars["bar_start"]
    )

    if not actual_duration.eq(expected_duration).all():
        raise AssertionError(
            f"{timeframe}: invalid bar duration."
        )

    if not (
        bars["available_time"].to_numpy()
        == bars.index.to_numpy()
    ).all():
        raise AssertionError(
            f"{timeframe}: available_time != bar_end."
        )

    if (bars["coverage_ratio"] <= 0).any():
        raise AssertionError(
            f"{timeframe}: non-positive coverage."
        )

    if (bars["coverage_ratio"] > 1).any():
        raise AssertionError(
            f"{timeframe}: coverage exceeds 1."
        )

    if (
        bars["high"]
        < bars[["open", "low", "close"]].max(axis=1)
    ).any():
        raise AssertionError(
            f"{timeframe}: invalid high."
        )

    if (
        bars["low"]
        > bars[["open", "high", "close"]].min(axis=1)
    ).any():
        raise AssertionError(
            f"{timeframe}: invalid low."
        )

In [13]:
# for timeframe, bars in bar_library.items():
#     validate_time_bars(
#         bars,
#         timeframe,
#     )

# print("All bar validations passed.")

In [14]:
# print(validation_report)

# for timeframe, bars in bar_library.items():
#     print(
#         timeframe,
#         bars.shape,
#         bars["is_complete"].mean(),
#         bars["coverage_ratio"].min(),
#         bars["coverage_ratio"].mean(),
#         bars.index.min(),
#         bars.index.max(),
#     )

# State Engine

## Function

In [15]:
from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Literal

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class RegimeConfig:
    regime_timeframe: str = "4h"
    state_family: str = "imom_imacd"

    atr_period: int = 14

    imacd_ma_len: int = 34
    imacd_signal_len: int = 9

    # absolute = ใช้ imacd_sb หน่วยราคาเหมือนโค้ดเดิม
    # atr      = ใช้ imacd_sb / ATR
    # close    = ใช้ imacd_sb / close
    imom_scale: Literal["absolute", "atr", "close"] = "absolute"
    imom_eps: float = 2.0

    persistence_min_len: int = 4

    # True = State แรกต้องปรากฏติดต่อกันครบ min_len ก่อน
    # False = State แรกถูกยืนยันทันทีเหมือนโค้ดเดิม
    confirm_initial_state: bool = True

    # ไม่พก State ข้ามช่วงเวลาที่ข้อมูล bars ขาด
    reset_after_time_gap: bool = True

    # เมื่อ raw state ใช้งานไม่ได้ ให้ State ที่ยืนยันแล้วเป็น NaN
    invalidate_on_missing_raw: bool = True

    # ตัวอย่าง 1.0 = ใช้เฉพาะ bar ที่มี minute ครบทั้งหมด
    # 0.0 = ใช้ทุก bar ที่มีข้อมูลอย่างน้อยหนึ่งนาที
    min_coverage_ratio: float = 0.0

In [16]:
BASE_REGIME_CONFIG = RegimeConfig(
    regime_timeframe="4h",

    atr_period=14,

    imacd_ma_len=34,
    imacd_signal_len=9,

    imom_scale="absolute",
    imom_eps=2.0,

    persistence_min_len=4,
    confirm_initial_state=True,

    min_coverage_ratio=0.0,
)

In [17]:
def rma(
    series: pd.Series,
    period: int,
) -> pd.Series:
    if period <= 0:
        raise ValueError("period must be positive.")

    return series.ewm(
        alpha=1.0 / period,
        adjust=False,
    ).mean()


def add_atr(
    df: pd.DataFrame,
    period: int = 14,
) -> pd.DataFrame:
    required = {"high", "low", "close"}

    missing = required.difference(df.columns)
    if missing:
        raise ValueError(
            f"Missing ATR columns: {sorted(missing)}"
        )

    d = df.copy()

    prev_close = d["close"].shift(1)

    tr_components = pd.concat(
        [
            d["high"] - d["low"],
            (d["high"] - prev_close).abs(),
            (d["low"] - prev_close).abs(),
        ],
        axis=1,
    )

    d["tr"] = tr_components.max(axis=1)
    d["atr"] = rma(d["tr"], period)

    return d

In [18]:
def smma(
    series: pd.Series,
    length: int,
) -> pd.Series:
    """
    SMMA:
    - ค่าแรกเริ่มจากค่าเฉลี่ย length จุด
    - หลังจากนั้นใช้ recursive smoothing
    - หากเจอ NaN จะไม่อัปเดตค่า internal state
    """

    if length <= 0:
        raise ValueError("length must be positive.")

    values = series.to_numpy(dtype=float)
    result = np.full(len(values), np.nan, dtype=float)

    if len(values) < length:
        return pd.Series(
            result,
            index=series.index,
            name=series.name,
        )

    initialized = False
    previous_value = np.nan

    for i in range(length - 1, len(values)):
        current_value = values[i]

        if not initialized:
            seed = values[i - length + 1 : i + 1]

            if np.isfinite(seed).all():
                previous_value = float(seed.mean())
                result[i] = previous_value
                initialized = True

            continue

        if not np.isfinite(current_value):
            result[i] = np.nan
            continue

        previous_value = (
            previous_value * (length - 1)
            + current_value
        ) / length

        result[i] = previous_value

    return pd.Series(
        result,
        index=series.index,
        name=series.name,
    )

In [19]:
def zlema(
    series: pd.Series,
    length: int,
) -> pd.Series:
    if length <= 0:
        raise ValueError("length must be positive.")

    ema1 = series.ewm(
        span=length,
        adjust=False,
    ).mean()

    ema2 = ema1.ewm(
        span=length,
        adjust=False,
    ).mean()

    return ema1 + (ema1 - ema2)

In [20]:
def add_imacd_features(
    bars: pd.DataFrame,
    cfg: RegimeConfig,
) -> pd.DataFrame:
    required = {"open", "high", "low", "close"}

    missing = required.difference(bars.columns)
    if missing:
        raise ValueError(
            f"Missing IMACD columns: {sorted(missing)}"
        )

    d = add_atr(
        bars,
        period=cfg.atr_period,
    )

    high = d["high"].astype(float)
    low = d["low"].astype(float)
    close = d["close"].astype(float)

    hlc3 = (
        high
        + low
        + close
    ) / 3.0

    imacd_hi = smma(
        high,
        cfg.imacd_ma_len,
    )

    imacd_lo = smma(
        low,
        cfg.imacd_ma_len,
    )

    imacd_mi = zlema(
        hlc3,
        cfg.imacd_ma_len,
    )

    imacd_md = pd.Series(
        np.where(
            imacd_mi > imacd_hi,
            imacd_mi - imacd_hi,
            np.where(
                imacd_mi < imacd_lo,
                imacd_mi - imacd_lo,
                0.0,
            ),
        ),
        index=d.index,
        dtype=float,
    )

    imacd_sb = imacd_md.rolling(
        window=cfg.imacd_signal_len,
        min_periods=cfg.imacd_signal_len,
    ).mean()

    imacd_sh = imacd_md - imacd_sb

    color = pd.Series(
        pd.NA,
        index=d.index,
        dtype="string",
    )

    feature_ready = (
        hlc3.notna()
        & imacd_hi.notna()
        & imacd_lo.notna()
        & imacd_mi.notna()
    )

    lime_mask = (
        feature_ready
        & (hlc3 > imacd_mi)
        & (hlc3 > imacd_hi)
    )

    green_mask = (
        feature_ready
        & (hlc3 > imacd_mi)
        & ~(hlc3 > imacd_hi)
    )

    red_mask = (
        feature_ready
        & ~(hlc3 > imacd_mi)
        & (hlc3 < imacd_lo)
    )

    orange_mask = (
        feature_ready
        & ~(hlc3 > imacd_mi)
        & ~(hlc3 < imacd_lo)
    )

    color.loc[lime_mask] = "lime"
    color.loc[green_mask] = "green"
    color.loc[red_mask] = "red"
    color.loc[orange_mask] = "orange"

    d["hlc3"] = hlc3

    d["imacd_hi"] = imacd_hi
    d["imacd_lo"] = imacd_lo
    d["imacd_mi"] = imacd_mi

    d["imacd_md"] = imacd_md
    d["imacd_sb"] = imacd_sb
    d["imacd_sh"] = imacd_sh

    d["imacd_color"] = color

    return d

In [21]:
def calculate_imom_score(
    df: pd.DataFrame,
    cfg: RegimeConfig,
) -> pd.Series:
    if cfg.imom_scale == "absolute":
        score = df["imacd_sb"]

    elif cfg.imom_scale == "atr":
        denominator = df["atr"].replace(0.0, np.nan)
        score = df["imacd_sb"] / denominator

    elif cfg.imom_scale == "close":
        denominator = df["close"].replace(0.0, np.nan)
        score = df["imacd_sb"] / denominator

    else:
        raise ValueError(
            f"Unknown imom_scale: {cfg.imom_scale}"
        )

    return score.astype(float)

In [22]:
def add_raw_imom_imacd_state(
    df: pd.DataFrame,
    cfg: RegimeConfig,
) -> pd.DataFrame:
    d = df.copy()

    d["imom_score"] = calculate_imom_score(
        d,
        cfg,
    )

    imom_label = pd.Series(
        pd.NA,
        index=d.index,
        dtype="string",
    )

    score_ready = d["imom_score"].notna()

    imom_label.loc[
        score_ready
        & (d["imom_score"] > cfg.imom_eps)
    ] = "plus"

    imom_label.loc[
        score_ready
        & (d["imom_score"] < -cfg.imom_eps)
    ] = "minus"

    imom_label.loc[
        score_ready
        & (d["imom_score"] <= cfg.imom_eps)
        & (d["imom_score"] >= -cfg.imom_eps)
    ] = "flat"

    d["imom_label"] = imom_label

    if "coverage_ratio" in d.columns:
        coverage_valid = (
            d["coverage_ratio"]
            >= cfg.min_coverage_ratio
        )
    else:
        coverage_valid = pd.Series(
            True,
            index=d.index,
        )

    d["state_input_valid"] = (
        coverage_valid
        & d["imom_label"].notna()
        & d["imacd_color"].notna()
    )

    raw_state = d["imom_label"].str.cat(
        d["imacd_color"],
        sep="_",
    )

    d["raw_state"] = raw_state.where(
        d["state_input_valid"],
        pd.NA,
    ).astype("string")

    return d

In [23]:
def enforce_persistence_with_diagnostics(
    raw_state: pd.Series,
    *,
    min_len: int,
    expected_step: pd.Timedelta,
    confirm_initial_state: bool = True,
    reset_after_time_gap: bool = True,
    invalidate_on_missing_raw: bool = True,
) -> pd.DataFrame:
    """
    ยืนยัน State แบบ causal

    ไม่มีการย้อนกลับไปแก้ State ในอดีต

    ตัวอย่าง min_len=4:

    raw:
        A A A A B B B B

    confirmed:
        - - - A A A A B

    State B มีผลเมื่อเห็น B ครบ 4 แท่งแล้วเท่านั้น
    """

    if min_len <= 0:
        raise ValueError("min_len must be positive.")

    if not isinstance(
        raw_state.index,
        pd.DatetimeIndex,
    ):
        raise TypeError(
            "raw_state index must be DatetimeIndex."
        )

    values = raw_state.astype("object").to_numpy()
    times = raw_state.index

    n = len(raw_state)

    confirmed_values = np.full(
        n,
        None,
        dtype=object,
    )

    candidate_values = np.full(
        n,
        None,
        dtype=object,
    )

    candidate_counts = np.zeros(
        n,
        dtype=np.int64,
    )

    state_ages = np.zeros(
        n,
        dtype=np.int64,
    )

    state_changes = np.zeros(
        n,
        dtype=bool,
    )

    state_ids = np.full(
        n,
        -1,
        dtype=np.int64,
    )

    gap_flags = np.zeros(
        n,
        dtype=bool,
    )

    current_state = None
    candidate_state = None
    candidate_count = 0

    current_state_age = 0
    current_state_id = 0

    previous_time = None

    for i in range(n):
        timestamp = times[i]
        value = values[i]

        has_time_gap = (
            previous_time is not None
            and timestamp - previous_time != expected_step
        )

        gap_flags[i] = has_time_gap

        if has_time_gap and reset_after_time_gap:
            current_state = None
            candidate_state = None
            candidate_count = 0
            current_state_age = 0

        previous_time = timestamp

        if pd.isna(value):
            candidate_state = None
            candidate_count = 0

            if invalidate_on_missing_raw:
                current_state = None
                current_state_age = 0

            if current_state is not None:
                current_state_age += 1

                confirmed_values[i] = current_state
                state_ages[i] = current_state_age
                state_ids[i] = current_state_id

            continue

        value = str(value)

        # ยังไม่มี State ที่ได้รับการยืนยัน
        if current_state is None:
            if not confirm_initial_state:
                current_state = value
                current_state_age = 1

                current_state_id += 1

                confirmed_values[i] = current_state
                state_ages[i] = current_state_age
                state_ids[i] = current_state_id
                state_changes[i] = True

                candidate_state = None
                candidate_count = 0

                continue

            if value == candidate_state:
                candidate_count += 1
            else:
                candidate_state = value
                candidate_count = 1

            if candidate_count >= min_len:
                current_state = candidate_state
                current_state_age = 1

                current_state_id += 1

                confirmed_values[i] = current_state
                state_ages[i] = current_state_age
                state_ids[i] = current_state_id
                state_changes[i] = True

                candidate_state = None
                candidate_count = 0

            else:
                candidate_values[i] = candidate_state
                candidate_counts[i] = candidate_count

            continue

        # Raw state ตรงกับ State ปัจจุบัน
        if value == current_state:
            candidate_state = None
            candidate_count = 0

            current_state_age += 1

            confirmed_values[i] = current_state
            state_ages[i] = current_state_age
            state_ids[i] = current_state_id

            continue

        # Raw state ต่างจาก State ปัจจุบัน
        if value == candidate_state:
            candidate_count += 1
        else:
            candidate_state = value
            candidate_count = 1

        # Candidate ครบจำนวนขั้นต่ำแล้ว
        if candidate_count >= min_len:
            current_state = candidate_state
            current_state_age = 1

            current_state_id += 1

            confirmed_values[i] = current_state
            state_ages[i] = current_state_age
            state_ids[i] = current_state_id
            state_changes[i] = True

            candidate_state = None
            candidate_count = 0

        else:
            # ระหว่างรอ candidate ยืนยัน
            # State เดิมยังคงมีผลอยู่
            current_state_age += 1

            confirmed_values[i] = current_state
            state_ages[i] = current_state_age
            state_ids[i] = current_state_id

            candidate_values[i] = candidate_state
            candidate_counts[i] = candidate_count

    result = pd.DataFrame(
        index=raw_state.index,
    )

    result["confirmed_state"] = pd.Series(
        confirmed_values,
        index=raw_state.index,
        dtype="string",
    )

    result["candidate_state"] = pd.Series(
        candidate_values,
        index=raw_state.index,
        dtype="string",
    )

    result["candidate_count"] = pd.Series(
        candidate_counts,
        index=raw_state.index,
        dtype="Int64",
    )

    result["state_age_bars"] = pd.Series(
        state_ages,
        index=raw_state.index,
        dtype="Int64",
    ).where(
        pd.Series(
            confirmed_values,
            index=raw_state.index,
        ).notna()
    )

    result["state_change"] = state_changes

    result["state_id"] = pd.Series(
        state_ids,
        index=raw_state.index,
        dtype="Int64",
    ).where(
        pd.Series(
            state_ids,
            index=raw_state.index,
        ) >= 0
    )

    result["time_gap_detected"] = gap_flags

    result["state_ready"] = (
        result["confirmed_state"].notna()
    )

    result["raw_disagrees_with_confirmed"] = (
        raw_state.notna()
        & result["confirmed_state"].notna()
        & (
            raw_state.astype("string")
            != result["confirmed_state"]
        )
    )

    return result

In [24]:
def build_imom_imacd_regime(
    bars: pd.DataFrame,
    cfg: RegimeConfig,
) -> pd.DataFrame:
    if not isinstance(
        bars.index,
        pd.DatetimeIndex,
    ):
        raise TypeError(
            "bars index must be DatetimeIndex."
        )

    if not bars.index.is_monotonic_increasing:
        raise ValueError(
            "bars must be sorted by time."
        )

    if bars.index.duplicated().any():
        raise ValueError(
            "bars contain duplicate timestamps."
        )

    d = add_imacd_features(
        bars,
        cfg,
    )

    d = add_raw_imom_imacd_state(
        d,
        cfg,
    )

    persistence = enforce_persistence_with_diagnostics(
        d["raw_state"],
        min_len=cfg.persistence_min_len,
        expected_step=pd.Timedelta(
            cfg.regime_timeframe
        ),
        confirm_initial_state=cfg.confirm_initial_state,
        reset_after_time_gap=cfg.reset_after_time_gap,
        invalidate_on_missing_raw=(
            cfg.invalidate_on_missing_raw
        ),
    )

    d = d.join(persistence)

    d["state_family"] = cfg.state_family
    d["regime_timeframe"] = cfg.regime_timeframe

    # bar index หมายถึงเวลาที่ข้อมูลพร้อมแล้ว
    d["available_time"] = d.index

    d["state_confirmation_time"] = pd.NaT
    d.loc[
        d["state_change"],
        "state_confirmation_time",
    ] = d.index[d["state_change"]]

    return d

In [25]:
def validate_regime_output(
    regime: pd.DataFrame,
    cfg: RegimeConfig,
) -> None:
    required = {
        "raw_state",
        "confirmed_state",
        "candidate_state",
        "candidate_count",
        "state_age_bars",
        "state_change",
        "state_id",
        "available_time",
    }

    missing = required.difference(regime.columns)

    if missing:
        raise AssertionError(
            f"Missing regime columns: {sorted(missing)}"
        )

    if not regime.index.is_monotonic_increasing:
        raise AssertionError(
            "Regime index is not monotonic."
        )

    if regime.index.duplicated().any():
        raise AssertionError(
            "Regime contains duplicated timestamps."
        )

    if not (
        regime["available_time"].to_numpy()
        == regime.index.to_numpy()
    ).all():
        raise AssertionError(
            "available_time must equal bar_end."
        )

    candidate_count = (
        regime["candidate_count"]
        .fillna(0)
        .astype(int)
    )

    if (
        candidate_count
        >= cfg.persistence_min_len
    ).any():
        raise AssertionError(
            "Candidate remained pending after reaching min_len."
        )

    change_rows = regime.loc[
        regime["state_change"]
    ]

    invalid_change = (
        change_rows["raw_state"]
        != change_rows["confirmed_state"]
    )

    if invalid_change.any():
        raise AssertionError(
            "Confirmed State does not match raw State "
            "on confirmation row."
        )

    ready_expected = (
        regime["confirmed_state"].notna()
    )

    if not (
        regime["state_ready"]
        == ready_expected
    ).all():
        raise AssertionError(
            "state_ready is inconsistent."
        )

    valid_age = regime.loc[
        regime["confirmed_state"].notna(),
        "state_age_bars",
    ]

    if (valid_age < 1).any():
        raise AssertionError(
            "Confirmed State must have age >= 1."
        )

In [26]:
def build_regime_library(
    bar_library: dict[str, pd.DataFrame],
    regime_configs: dict[str, RegimeConfig],
) -> dict[str, pd.DataFrame]:

    result: dict[str, pd.DataFrame] = {}

    for timeframe, cfg in regime_configs.items():
        if timeframe not in bar_library:
            raise KeyError(
                f"No bars found for timeframe: {timeframe}"
            )

        regime = build_imom_imacd_regime(
            bar_library[timeframe],
            cfg,
        )

        validate_regime_output(
            regime,
            cfg,
        )

        result[timeframe] = regime

    return result

In [27]:
def extract_state_segments(
    regime: pd.DataFrame,
    timeframe: str,
) -> pd.DataFrame:
    required = {
        "state_id",
        "confirmed_state",
    }

    missing = required.difference(regime.columns)

    if missing:
        raise ValueError(
            f"Missing columns: {sorted(missing)}"
        )

    valid = regime.loc[
        regime["state_id"].notna()
        & regime["confirmed_state"].notna()
    ].copy()

    if valid.empty:
        return pd.DataFrame(
            columns=[
                "state_id",
                "confirmed_state",
                "start_time",
                "end_time",
                "n_bars",
                "duration",
            ]
        )

    grouped = valid.groupby(
        "state_id",
        sort=True,
        observed=True,
    )

    segments = grouped.agg(
        confirmed_state=(
            "confirmed_state",
            "first",
        ),
        start_time=(
            "available_time",
            "min",
        ),
        end_time=(
            "available_time",
            "max",
        ),
        n_bars=(
            "confirmed_state",
            "size",
        ),
    ).reset_index()

    bar_duration = pd.Timedelta(timeframe)

    segments["duration"] = (
        segments["end_time"]
        - segments["start_time"]
        + bar_duration
    )

    return segments

## Runner

In [28]:
# for timeframe, regime in regime_library.items():
#     print("=" * 80)
#     print("Timeframe:", timeframe)
#     print("Rows:", len(regime))
#     print(
#         "State-ready ratio:",
#         regime["state_ready"].mean(),
#     )
#     print(
#         "Number of confirmed changes:",
#         regime["state_change"].sum(),
#     )

#     display(
#         regime[
#             [
#                 "close",
#                 "imacd_sb",
#                 "imom_score",
#                 "imom_label",
#                 "imacd_color",
#                 "raw_state",
#                 "candidate_state",
#                 "candidate_count",
#                 "confirmed_state",
#                 "state_change",
#                 "state_age_bars",
#                 "state_id",
#             ]
#         ].tail(20)
#     )

In [29]:
# for timeframe, regime in regime_library.items():
#     state_distribution = (
#         regime["confirmed_state"]
#         .value_counts(
#             normalize=False,
#             dropna=False,
#         )
#         .rename("n_bars")
#         .to_frame()
#     )

#     state_distribution["ratio"] = (
#         state_distribution["n_bars"]
#         / len(regime)
#     )

#     print(f"\nTimeframe: {timeframe}")
#     display(state_distribution)

In [30]:
# for timeframe, segments in segment_library.items():
#     print("=" * 80)
#     print("Timeframe:", timeframe)
#     print("Number of segments:", len(segments))

#     display(
#         segments.groupby(
#             "confirmed_state",
#             observed=True,
#         ).agg(
#             n_segments=("state_id", "count"),
#             mean_bars=("n_bars", "mean"),
#             median_bars=("n_bars", "median"),
#             max_bars=("n_bars", "max"),
#         ).sort_values(
#             "n_segments",
#             ascending=False,
#         )
#     )

# Point-in-Time Alignment Engine

In [31]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class AlignmentConfig:
    regime_timeframe: str
    execution_timeframe: str

    state_family: str = "imom_imacd"

    # True:
    # State ที่ available เวลา 04:00 ใช้ตัดสินใจเวลา 04:00 ได้
    allow_exact_match: bool = True

    # ถ้าแท่ง Regime ถัดไปควรมาถึงแล้วแต่ไม่มีข้อมูล
    # จะไม่ถือ State เก่าต่อไปเรื่อย ๆ
    expire_after_missing_update: bool = True

    # ถ้าระบุ จะใช้แทน regime_timeframe
    # เช่น "8h" หมายถึงยอมให้ State เก่าได้สูงสุด 8 ชั่วโมง
    max_staleness: Optional[str] = None

In [32]:
# alignment_cfg = AlignmentConfig(
#     regime_timeframe="4h",
#     execution_timeframe="15min",
#     allow_exact_match=True,
#     expire_after_missing_update=True,
# )

In [33]:
def normalize_column_token(value: str) -> str:
    return (
        str(value)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("/", "_")
        .replace("-", "_")
    )


def make_state_prefix(
    state_family: str,
    regime_timeframe: str,
) -> str:
    family = normalize_column_token(state_family)
    timeframe = normalize_column_token(regime_timeframe)

    return f"{family}_{timeframe}"

In [34]:
make_state_prefix("imom_imacd", "4h")

'imom_imacd_4h'

In [35]:
def add_point_in_time_state_metadata(
    regime: pd.DataFrame,
) -> pd.DataFrame:
    required = {
        "available_time",
        "confirmed_state",
        "state_id",
    }

    missing = required.difference(regime.columns)

    if missing:
        raise ValueError(
            f"Missing regime columns: {sorted(missing)}"
        )

    d = regime.copy()

    d["available_time"] = pd.to_datetime(
        d["available_time"],
        utc=True,
    )

    d["state_start_time"] = pd.NaT

    valid_state = (
        d["state_id"].notna()
        & d["confirmed_state"].notna()
    )

    if valid_state.any():
        state_start = (
            d.loc[valid_state]
            .groupby(
                "state_id",
                sort=False,
                observed=True,
            )["available_time"]
            .transform("min")
        )

        d.loc[
            valid_state,
            "state_start_time",
        ] = state_start

    d["state_start_time"] = pd.to_datetime(
        d["state_start_time"],
        utc=True,
    )

    return d

In [36]:
def prepare_regime_for_alignment(
    regime: pd.DataFrame,
    cfg: AlignmentConfig,
) -> tuple[pd.DataFrame, str]:
    if not isinstance(regime.index, pd.DatetimeIndex):
        raise TypeError(
            "Regime index must be DatetimeIndex."
        )

    if not regime.index.is_monotonic_increasing:
        raise ValueError(
            "Regime must be sorted by time."
        )

    if regime.index.duplicated().any():
        raise ValueError(
            "Regime contains duplicated timestamps."
        )

    d = add_point_in_time_state_metadata(regime)

    prefix = make_state_prefix(
        cfg.state_family,
        cfg.regime_timeframe,
    )

    d["source_bar_end"] = pd.to_datetime(
        d.index,
        utc=True,
    )

    candidate_columns = [
        "source_bar_end",
        "available_time",
        "state_start_time",

        "raw_state",
        "confirmed_state",

        "candidate_state",
        "candidate_count",

        "state_age_bars",
        "state_id",
        "state_ready",

        "state_input_valid",
        "raw_disagrees_with_confirmed",
        "time_gap_detected",

        "imom_score",
        "imom_label",
        "imacd_color",
    ]

    available_columns = [
        col
        for col in candidate_columns
        if col in d.columns
    ]

    d = d[available_columns].copy()

    rename_map = {}

    for col in d.columns:
        if col == "available_time":
            rename_map[col] = (
                f"{prefix}_source_available_time"
            )
        else:
            rename_map[col] = f"{prefix}_{col}"

    d = d.rename(columns=rename_map)

    source_time_col = (
        f"{prefix}_source_available_time"
    )

    d = d.sort_values(source_time_col)

    return d, prefix

In [37]:
def align_regime_to_execution(
    execution_bars: pd.DataFrame,
    regime: pd.DataFrame,
    cfg: AlignmentConfig,
) -> pd.DataFrame:
    """
    นำ State จาก regime timeframe ไป align กับ execution bars

    กฎ:
    source_available_time <= decision_time

    ไม่มี execution row ใดเห็น State จากอนาคต
    """

    if not isinstance(
        execution_bars.index,
        pd.DatetimeIndex,
    ):
        raise TypeError(
            "Execution bars index must be DatetimeIndex."
        )

    if not execution_bars.index.is_monotonic_increasing:
        raise ValueError(
            "Execution bars must be sorted by time."
        )

    if execution_bars.index.duplicated().any():
        raise ValueError(
            "Execution bars contain duplicated timestamps."
        )

    left = execution_bars.copy()

    # เก็บเวลาปิดของ execution bar ไว้ชัดเจน
    left["execution_bar_end"] = pd.to_datetime(
        left.index,
        utc=True,
    )

    # จุดที่ strategy รู้ข้อมูลของ execution bar ครบ
    if "available_time" in left.columns:
        left["decision_time"] = pd.to_datetime(
            left["available_time"],
            utc=True,
        )
    else:
        left["decision_time"] = left[
            "execution_bar_end"
        ]

    right, prefix = prepare_regime_for_alignment(
        regime,
        cfg,
    )

    source_time_col = (
        f"{prefix}_source_available_time"
    )

    new_columns = set(right.columns).union(
        {
            f"{prefix}_alignment_valid",
            f"{prefix}_state_is_stale",
            f"{prefix}_is_regime_update",
            f"{prefix}_aligned_state_change",
            f"{prefix}_state_elapsed_time",
            f"{prefix}_state_ready_for_decision",
        }
    )

    duplicated_new_columns = (
        new_columns.intersection(left.columns)
    )

    if duplicated_new_columns:
        raise ValueError(
            "Alignment columns already exist: "
            f"{sorted(duplicated_new_columns)}"
        )

    left = (
        left
        .reset_index(drop=True)
        .sort_values("decision_time")
    )

    aligned = pd.merge_asof(
        left=left,
        right=right,
        left_on="decision_time",
        right_on=source_time_col,
        direction="backward",
        allow_exact_matches=cfg.allow_exact_match,
    )

    source_time = aligned[source_time_col]
    decision_time = aligned["decision_time"]

    # ตรวจว่ามี State ต้นทางให้ match หรือไม่
    source_found = source_time.notna()

    if cfg.max_staleness is not None:
        allowed_staleness = pd.Timedelta(
            cfg.max_staleness
        )

    elif cfg.expire_after_missing_update:
        allowed_staleness = pd.Timedelta(
            cfg.regime_timeframe
        )

    else:
        allowed_staleness = None

    if allowed_staleness is None:
        state_is_stale = pd.Series(
            False,
            index=aligned.index,
        )
    else:
        next_expected_update = (
            source_time + allowed_staleness
        )

        # ที่เวลา next expected update:
        # ถ้าแท่งใหม่มีจริง merge_asof จะ match แท่งใหม่
        # ถ้าไม่มี จะยัง match แท่งเก่าและถูกระบุว่า stale
        state_is_stale = (
            source_found
            & (decision_time >= next_expected_update)
        )

    alignment_valid = (
        source_found
        & ~state_is_stale
    )

    aligned[
        f"{prefix}_state_is_stale"
    ] = state_is_stale

    aligned[
        f"{prefix}_alignment_valid"
    ] = alignment_valid

    protected_source_columns = {
        source_time_col,
        f"{prefix}_source_bar_end",
    }

    # เมื่อ State stale:
    # ไม่อนุญาตให้ strategy เห็น State หรือ diagnostics เก่า
    payload_columns = [
        col
        for col in right.columns
        if col not in protected_source_columns
    ]

    for col in payload_columns:
        aligned.loc[
            ~alignment_valid,
            col,
        ] = pd.NA

    confirmed_state_col = (
        f"{prefix}_confirmed_state"
    )

    state_id_col = (
        f"{prefix}_state_id"
    )

    state_start_col = (
        f"{prefix}_state_start_time"
    )

    # มี State พร้อมใช้จริง ณ decision time
    state_ready_for_decision = (
        alignment_valid
        & aligned[confirmed_state_col].notna()
    )

    aligned[
        f"{prefix}_state_ready_for_decision"
    ] = state_ready_for_decision

    # Execution bar นี้ตรงกับเวลาที่ Regime bar ใหม่ปิดพอดีหรือไม่
    aligned[
        f"{prefix}_is_regime_update"
    ] = (
        alignment_valid
        & source_time.eq(decision_time)
    )

    # State เปลี่ยนจากมุมมองของ execution timeline
    aligned_state_id = aligned[
        state_id_col
    ]

    aligned[
        f"{prefix}_aligned_state_change"
    ] = (
        aligned_state_id.notna()
        & aligned_state_id.ne(
            aligned_state_id.shift(1)
        )
    )

    # อายุ State เป็นเวลาแท้จริง
    aligned[
        f"{prefix}_state_elapsed_time"
    ] = (
        decision_time
        - pd.to_datetime(
            aligned[state_start_col],
            utc=True,
        )
    )

    aligned = (
        aligned
        .sort_values("execution_bar_end")
        .set_index("execution_bar_end")
    )

    aligned.index.name = "bar_end"

    return aligned

In [38]:
def validate_regime_alignment(
    aligned: pd.DataFrame,
    regime: pd.DataFrame,
    cfg: AlignmentConfig,
) -> None:
    prefix = make_state_prefix(
        cfg.state_family,
        cfg.regime_timeframe,
    )

    required = {
        "decision_time",
        f"{prefix}_source_available_time",
        f"{prefix}_alignment_valid",
        f"{prefix}_state_is_stale",
        f"{prefix}_confirmed_state",
        f"{prefix}_state_ready_for_decision",
        f"{prefix}_aligned_state_change",
    }

    missing = required.difference(
        aligned.columns
    )

    if missing:
        raise AssertionError(
            f"Missing alignment columns: {sorted(missing)}"
        )

    decision_time = pd.to_datetime(
        aligned["decision_time"],
        utc=True,
    )

    source_time = pd.to_datetime(
        aligned[
            f"{prefix}_source_available_time"
        ],
        utc=True,
    )

    matched = source_time.notna()

    if cfg.allow_exact_match:
        future_source = (
            matched
            & (source_time > decision_time)
        )
    else:
        future_source = (
            matched
            & (source_time >= decision_time)
        )

    if future_source.any():
        bad_rows = aligned.loc[
            future_source,
            [
                "decision_time",
                f"{prefix}_source_available_time",
            ],
        ].head(10)

        raise AssertionError(
            "Future Regime State detected.\n"
            f"{bad_rows}"
        )

    valid_col = (
        f"{prefix}_alignment_valid"
    )

    stale_col = (
        f"{prefix}_state_is_stale"
    )

    if (
        aligned[valid_col]
        & aligned[stale_col]
    ).any():
        raise AssertionError(
            "A State cannot be valid and stale simultaneously."
        )

    ready_col = (
        f"{prefix}_state_ready_for_decision"
    )

    confirmed_col = (
        f"{prefix}_confirmed_state"
    )

    invalid_ready = (
        aligned[ready_col]
        & (
            ~aligned[valid_col]
            | aligned[confirmed_col].isna()
        )
    )

    if invalid_ready.any():
        raise AssertionError(
            "state_ready_for_decision is inconsistent."
        )

    change_col = (
        f"{prefix}_aligned_state_change"
    )

    if (
        aligned[change_col]
        & ~aligned[ready_col]
    ).any():
        raise AssertionError(
            "Aligned State changed while State was not ready."
        )

    elapsed_col = (
        f"{prefix}_state_elapsed_time"
    )

    negative_elapsed = (
        aligned[elapsed_col].notna()
        & (
            aligned[elapsed_col]
            < pd.Timedelta(0)
        )
    )

    if negative_elapsed.any():
        raise AssertionError(
            "Negative State elapsed time detected."
        )

    # source_time ที่ align มา ต้องมีอยู่จริงใน Regime table
    actual_source_times = set(
        pd.to_datetime(
            regime["available_time"],
            utc=True,
        )
    )

    unknown_source = (
        matched
        & ~source_time.isin(actual_source_times)
    )

    if unknown_source.any():
        raise AssertionError(
            "Aligned source time does not exist "
            "in Regime table."
        )

In [39]:
def build_execution_state_dataset(
    execution_bars: pd.DataFrame,
    execution_timeframe: str,
    regime_library: dict[str, pd.DataFrame],
    regime_timeframes: list[str],
    *,
    state_family: str = "imom_imacd",
    allow_exact_match: bool = True,
    expire_after_missing_update: bool = True,
) -> pd.DataFrame:
    result = execution_bars.copy()

    for regime_timeframe in regime_timeframes:
        if regime_timeframe not in regime_library:
            raise KeyError(
                "Missing Regime timeframe: "
                f"{regime_timeframe}"
            )

        cfg = AlignmentConfig(
            regime_timeframe=regime_timeframe,
            execution_timeframe=execution_timeframe,
            state_family=state_family,
            allow_exact_match=allow_exact_match,
            expire_after_missing_update=(
                expire_after_missing_update
            ),
        )

        result = align_regime_to_execution(
            execution_bars=result,
            regime=regime_library[
                regime_timeframe
            ],
            cfg=cfg,
        )

        validate_regime_alignment(
            aligned=result,
            regime=regime_library[
                regime_timeframe
            ],
            cfg=cfg,
        )

    return result

In [40]:
def build_full_execution_library(
    bar_library: dict[str, pd.DataFrame],
    regime_library: dict[str, pd.DataFrame],
    execution_timeframes: list[str],
    regime_timeframes: list[str],
) -> dict[str, pd.DataFrame]:
    result = {}

    for execution_timeframe in execution_timeframes:
        if execution_timeframe not in bar_library:
            raise KeyError(
                "Missing execution bars: "
                f"{execution_timeframe}"
            )

        execution_df = build_execution_state_dataset(
            execution_bars=bar_library[
                execution_timeframe
            ],
            execution_timeframe=execution_timeframe,
            regime_library=regime_library,
            regime_timeframes=regime_timeframes,
        )

        result[execution_timeframe] = (
            execution_df
        )

    return result

In [41]:
def inspect_market_state(
    execution_df: pd.DataFrame,
    timestamp: str | pd.Timestamp,
    *,
    state_family: str = "imom_imacd",
    regime_timeframes: list[str],
) -> pd.DataFrame:
    timestamp = pd.Timestamp(
        timestamp,
        tz="UTC",
    )

    if timestamp not in execution_df.index:
        raise KeyError(
            f"No execution bar at {timestamp}"
        )

    rows = []

    for timeframe in regime_timeframes:
        prefix = make_state_prefix(
            state_family,
            timeframe,
        )

        rows.append(
            {
                "regime_timeframe": timeframe,

                "source_available_time": (
                    execution_df.at[
                        timestamp,
                        f"{prefix}_source_available_time",
                    ]
                ),

                "confirmed_state": (
                    execution_df.at[
                        timestamp,
                        f"{prefix}_confirmed_state",
                    ]
                ),

                "state_ready": (
                    execution_df.at[
                        timestamp,
                        f"{prefix}_state_ready_for_decision",
                    ]
                ),

                "state_elapsed_time": (
                    execution_df.at[
                        timestamp,
                        f"{prefix}_state_elapsed_time",
                    ]
                ),

                "alignment_valid": (
                    execution_df.at[
                        timestamp,
                        f"{prefix}_alignment_valid",
                    ]
                ),
            }
        )

    return pd.DataFrame(rows)

In [42]:
def test_basic_asof_alignment() -> None:
    execution_index = pd.date_range(
        "2025-01-01 03:45:00",
        "2025-01-01 08:15:00",
        freq="15min",
        tz="UTC",
    )

    execution = pd.DataFrame(
        {
            "open": 100.0,
            "high": 101.0,
            "low": 99.0,
            "close": 100.0,
            "available_time": execution_index,
        },
        index=execution_index,
    )

    regime_index = pd.DatetimeIndex(
        [
            "2025-01-01 04:00:00+00:00",
            "2025-01-01 08:00:00+00:00",
        ]
    )

    regime = pd.DataFrame(
        {
            "available_time": regime_index,
            "raw_state": [
                "plus_lime",
                "minus_red",
            ],
            "confirmed_state": [
                "plus_lime",
                "minus_red",
            ],
            "state_id": [1, 2],
            "state_ready": [True, True],
        },
        index=regime_index,
    )

    cfg = AlignmentConfig(
        regime_timeframe="4h",
        execution_timeframe="15min",
    )

    result = align_regime_to_execution(
        execution,
        regime,
        cfg,
    )

    state_col = (
        "imom_imacd_4h_confirmed_state"
    )

    assert pd.isna(
        result.loc[
            "2025-01-01 03:45:00+00:00",
            state_col,
        ]
    )

    assert (
        result.loc[
            "2025-01-01 04:00:00+00:00",
            state_col,
        ]
        == "plus_lime"
    )

    assert (
        result.loc[
            "2025-01-01 07:45:00+00:00",
            state_col,
        ]
        == "plus_lime"
    )

    assert (
        result.loc[
            "2025-01-01 08:00:00+00:00",
            state_col,
        ]
        == "minus_red"
    )

    print("Basic as-of alignment test passed.")

In [43]:
# test_basic_asof_alignment()

In [44]:
def test_state_expiration() -> None:
    execution_index = pd.date_range(
        "2025-01-01 04:00:00",
        "2025-01-01 09:00:00",
        freq="15min",
        tz="UTC",
    )

    execution = pd.DataFrame(
        {
            "open": 100.0,
            "high": 101.0,
            "low": 99.0,
            "close": 100.0,
            "available_time": execution_index,
        },
        index=execution_index,
    )

    # มีเฉพาะแท่ง 04:00
    # แท่งที่ควรปิด 08:00 หาย
    regime_index = pd.DatetimeIndex(
        ["2025-01-01 04:00:00+00:00"]
    )

    regime = pd.DataFrame(
        {
            "available_time": regime_index,
            "raw_state": ["plus_lime"],
            "confirmed_state": ["plus_lime"],
            "state_id": [1],
            "state_ready": [True],
        },
        index=regime_index,
    )

    cfg = AlignmentConfig(
        regime_timeframe="4h",
        execution_timeframe="15min",
        expire_after_missing_update=True,
    )

    result = align_regime_to_execution(
        execution,
        regime,
        cfg,
    )

    state_col = (
        "imom_imacd_4h_confirmed_state"
    )

    assert (
        result.loc[
            "2025-01-01 07:45:00+00:00",
            state_col,
        ]
        == "plus_lime"
    )

    assert pd.isna(
        result.loc[
            "2025-01-01 08:00:00+00:00",
            state_col,
        ]
    )

    assert bool(
        result.loc[
            "2025-01-01 08:00:00+00:00",
            "imom_imacd_4h_state_is_stale",
        ]
    )

    print("State expiration test passed.")

In [45]:
# test_state_expiration()